<a href="https://colab.research.google.com/github/brgael/GTD2401-ACD-GA-AD26-07-ml-intermedio/blob/main/02.%20ValoresExtraviados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Valores Extraviados

En este JNB se revisará el tema de Valores Extraviados.

In [ ]:
# Code you have previously used to load data
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data directly from URL
url = 'https://raw.githubusercontent.com/humbertgaar/ml1-unisabana-datasets/main/melb_data.csv'
data = pd.read_csv(url)

# Create target object and call it y
y = data.Price

# Create X
X = data.drop(['Price'], axis=1)

# Split into validation and training data
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=0
)

print("Datos cargados correctamente")
print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_valid:", X_valid.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [ ]:
# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns
                     if X_train[col].isnull().any()]

# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)

# Convert categorical columns into numerical columns
reduced_X_train = pd.get_dummies(reduced_X_train)
reduced_X_valid = pd.get_dummies(reduced_X_valid)

# Align columns in training and validation data
reduced_X_train, reduced_X_valid = reduced_X_train.align(
    reduced_X_valid, join='left', axis=1, fill_value=0
)

print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop columns with missing values):


In [ ]:
from sklearn.impute import SimpleImputer

# Imputation
numerical_X_train = X_train.select_dtypes(exclude=['object'])
numerical_X_valid = X_valid.select_dtypes(exclude=['object'])

my_imputer = SimpleImputer()

imputed_X_train = pd.DataFrame(
    my_imputer.fit_transform(numerical_X_train)
)

imputed_X_valid = pd.DataFrame(
    my_imputer.transform(numerical_X_valid)
)

# Imputation removed column names; put them back
imputed_X_train.columns = numerical_X_train.columns
imputed_X_valid.columns = numerical_X_valid.columns

print("MAE from Approach 2 (Imputation):")
print(score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

In [ ]:
# Make copy to avoid changing original data (when imputing)
X_train_plus = X_train.select_dtypes(exclude=['object']).copy()
X_valid_plus = X_valid.select_dtypes(exclude=['object']).copy()

# Get names of columns with missing values
cols_with_missing = [col for col in X_train_plus.columns
                     if X_train_plus[col].isnull().any()]

# Make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

# Imputation
my_imputer = SimpleImputer()

imputed_X_train_plus = pd.DataFrame(
    my_imputer.fit_transform(X_train_plus)
)

imputed_X_valid_plus = pd.DataFrame(
    my_imputer.transform(X_valid_plus)
)

# Imputation removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

print("MAE from Approach 3 (An Extension to Imputation):")
print(score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid))

In [ ]:
# Shape of training data (num_rows, num_columns)
print(X_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column = (X_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])